Questo notebook serve solo per far vedere come funzionano e come creare un dataset per allenare i modelli con hugging face.

# Preliminari

In [1]:
import os, re
import torch
import json
import copy
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
from huggingface_hub import login

from pathlib import Path


In [2]:
# Log in huggingface
load_dotenv()  # Load environment variables from .env file
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)
#if "COLAB_" not in "".join(os.environ.keys()):
#    from google.colab import userdata
#    hf_token = userdata.get('HF_TOKEN')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Parametri

In [3]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
DATA_FILE_NAME = "data_training.json"
CHAT_TEMPLATE = "llama-3.1"

## Load tokenizer

In [4]:
# Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
#messages = [
#    {"role": "user", "content": "Who are you?"},
#]
#inputs = tokenizer.apply_chat_template(
#	messages,
#	add_generation_prompt=True,
#	tokenize=True,
#	return_dict=True,
#	return_tensors="pt",
#).to(model.device)

#outputs = model.generate(**inputs, max_new_tokens=40)
#print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

# Data preparation

## Caricamento dei dati

In [5]:
# Carichiamo il nostro file JSON
if "COLAB_" not in "".join(os.environ.keys()):
    data_path = Path('../data/' + DATA_FILE_NAME)
else:
    data_path = DATA_FILE_NAME


with open(data_path, "r") as f:
    raw_json_data = json.load(f)

I dati in `raw_json_data` sono salvati con la seguente struttura:
```
{
    "conversations": [
        {
            "messages": [
                {"role": "system", "content": <contenuto di sistema>},
                {"role": "user", "content": <contenuto di user>},
                {"role": "assistant", "content": <contenuto di assistant>}
            ]
        },
        ...
    ]
}
```

Al momento `<contenuto di assistant>` è un dizionario e non è una stringa! Questo non ci permette di creare il dataset utilizzando la funzione `load_dataset`. Infatti, se si provasse con il seguente pezzo di codice:
```
dataset = Dataset.from_dict(raw_json_data)
```
si otterrebbe un errore.

Trasformiamo il contenuto dell'assistente in una stringha, creando un secondo dataset, chiamato `training_data`.

In [6]:
training_data = copy.deepcopy(raw_json_data)
for conversation in training_data['conversations']:
    for message in conversation['messages']:
        if message['role'] == 'assistant':
            message['content'] = json.dumps(message['content'])

Notiamo ora la differenzza:

In [7]:
# raw_json_data
print(type(raw_json_data['conversations'][0]['messages'][2]['content']))

<class 'dict'>


In [8]:
# training_data
print(type(training_data['conversations'][0]['messages'][2]['content']))

<class 'str'>


Possiamo ora creare il dataset con `load_dataset`

In [9]:
dataset = Dataset.from_dict(training_data)
print(dataset)
display(dataset.features)

Dataset({
    features: ['conversations'],
    num_rows: 173
})


{'conversations': {'messages': [{'content': Value(dtype='string', id=None),
    'role': Value(dtype='string', id=None)}]}}

In [10]:
# Analizziamo il primo record
display(dataset[0]['conversations']['messages'])

[{'content': "Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.",
  'role': 'system'},
 {'content': "RM ADDOME INFERIORE (S/C MDC)\r\n\r\n\r\nL'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .\r\n\r\nIN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.\r\nADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOADENOMEGALIA SEDE DI MALATTIA CON ASSE CORTO DI CIRCA 8 MM . ALTRI LINFONODI SOSPETTI SONO PRESENTI NEL MESO

## Parentesi sui chat templates

`Llama-3.1` e `Llama-3.2` usano il seguente formato per allenare i modelli:

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

CONTENUTO DI SISTEMA<|eot_id|><|start_header_id|>user<|end_header_id|>

CONTENUTO UTENTE<|eot_id|><|start_header_id|>assistant<|end_header_id|>

CONTENUTO ASSISTENTE<|eot_id|>
```

In [11]:
test_messages = [
    {'role': 'system', 'content': 'You are a helpful, respectful and honest assistant.'},
    {'role': 'user', 'content': 'Who are you?'},
    {'role': 'assistant', 'content': 'I am a large language model trained by Meta.'}
]

In [12]:
print(tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=False))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Sep 2025

You are a helpful, respectful and honest assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I am a large language model trained by Meta.<|eot_id|>


Non sempre è utile settare `add_generation_prompt=True`, come l'0esempio sottostante dimostra:

In [13]:
print(tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Sep 2025

You are a helpful, respectful and honest assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I am a large language model trained by Meta.<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## Creazione nuova colonna

Noi vogliamo aggiungere la colonna "text" al nostro `dataset` che sia di questo tipo:
```
{
    "text": [
        <testo 1>,
        <testo_2>,
        ...
    ]
}
```
cioè popolato da stringhe formattate con il chat template del modello.

Dobbiamo passare dal generico formato di Huggingface `("role", "content")` al formato specifico di Llama.

In [14]:
# Creiamo la funzione da applicare successivamente al dataset
def formatting_prompts_func(data):
    conversations = data["conversations"]
    texts = [tokenizer.apply_chat_template(conversation['messages'], tokenize=False, add_generation_prompt=False) for conversation in conversations]
    return {"text" : texts}

In [15]:
# Applichiamo la funzione al datset
dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/173 [00:00<?, ? examples/s]

In [16]:
# Ora abbiamo creato correttamente il nuovo campo "text"
print(dataset)
display(dataset.features)

Dataset({
    features: ['conversations', 'text'],
    num_rows: 173
})


{'conversations': {'messages': [{'content': Value(dtype='string', id=None),
    'role': Value(dtype='string', id=None)}]},
 'text': Value(dtype='string', id=None)}

In [17]:
# Osserviamo ora il primo record del dataset
display(dataset[0])

{'conversations': {'messages': [{'content': "Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.",
    'role': 'system'},
   {'content': "RM ADDOME INFERIORE (S/C MDC)\r\n\r\n\r\nL'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .\r\n\r\nIN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.\r\nADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOADENOMEGALIA SEDE DI MALATTIA CON ASSE CORTO DI CIRCA 8 MM . ALTRI LINFON

Vediamo come il chat template ha trasformato le conversazioni.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

In [18]:
print(dataset[0]['text'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Sep 2025

Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.<|eot_id|><|start_header_id|>user<|end_header_id|>

RM ADDOME INFERIORE (S/C MDC)


L'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .

IN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.
ADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOAD

## Aggiugiamo un ID

In [19]:
def add_id_column(batch, indices):
    return {"id": indices}

In [20]:
dataset = dataset.map(add_id_column, with_indices=True, batched=True)

Map:   0%|          | 0/173 [00:00<?, ? examples/s]

In [21]:
print(dataset)
display(dataset.features)

Dataset({
    features: ['conversations', 'text', 'id'],
    num_rows: 173
})


{'conversations': {'messages': [{'content': Value(dtype='string', id=None),
    'role': Value(dtype='string', id=None)}]},
 'text': Value(dtype='string', id=None),
 'id': Value(dtype='int64', id=None)}

# Splittiamo

In [22]:
splitted = dataset.train_test_split(test_size=0.2, seed=2025)
print(splitted)

DatasetDict({
    train: Dataset({
        features: ['conversations', 'text', 'id'],
        num_rows: 138
    })
    test: Dataset({
        features: ['conversations', 'text', 'id'],
        num_rows: 35
    })
})


In [23]:
print(splitted['train'][:10]['id'])
print(splitted['test'][:10]['id'])

[71, 29, 83, 72, 141, 118, 95, 0, 164, 30]
[142, 126, 134, 78, 156, 88, 96, 35, 129, 32]
